# ESZA019 — Visão Computacional
## Laboratório 8 — Rastreamento de Objetos (*Object Tracking*)

**Universidade Federal do ABC (UFABC)** — Engenharia de Instrumentação, Automação e Robótica

---

### Autores

| Nome completo | RA |
|---|---|
| João Vitor De Oliveira Lamano | — |
| Antonio Carlos de Freitas Vidal Junior | 11201920894 |
| Willian Kenji Takaracy | 11201812251 |

**Data de realização dos experimentos:** julho de 2026  
**Data de publicação do relatório:** julho de 2026

## 1. Introdução

Neste relatório documentamos o oitavo laboratório da disciplina de Visão Computacional, cujo
tema é o **rastreamento de objetos** (*object tracking*) em sequências de vídeo. O objetivo é
acompanhar, quadro a quadro, a posição de um objeto de interesse previamente delimitado por uma
região retangular (ROI, *Region of Interest*) selecionada manualmente pelo usuário.

Foram implementados dois programas em Python utilizando a biblioteca **OpenCV**:

1. **Experimento 1** — leitura de arquivos de vídeo (vídeos gravados pelos membros da equipe e os
   vídeos produzidos no *trabalho de vídeo* da disciplina). Para cada arquivo, o usuário seleciona
   manualmente a ROI do objeto a ser rastreado, o resultado é exibido em tela e o vídeo anotado é
   gravado em disco.

2. **Experimento 2** — versão adaptada do Experimento 1 que captura imagens diretamente da **webcam**.
   Além de rastrear o objeto selecionado, o programa exibe uma janela ao vivo com a imagem da câmera
   e a caixa delimitadora resultante, gravando também o vídeo de saída.

Ao longo do relatório apresentamos a fundamentação teórica dos rastreadores clássicos disponíveis no
OpenCV (BOOSTING, MIL, KCF, TLD, MedianFlow, MOSSE e CSRT) e do rastreador baseado em aprendizado
profundo **GOTURN**, descrevemos os procedimentos experimentais, o código desenvolvido, e discutimos
os resultados obtidos.

## 2. Fundamentação Teórica

### 2.1. O problema do rastreamento

Dado um objeto delimitado por uma caixa retangular em um quadro inicial de um vídeo, o problema de
rastreamento consiste em **estimar a posição (e frequentemente a escala) desse mesmo objeto nos
quadros subsequentes**. Diferentemente da *detecção*, que localiza objetos de forma independente em
cada quadro, o rastreamento explora a **coerência temporal** da cena: a posição do objeto no quadro
atual está fortemente correlacionada com a sua posição no quadro anterior. Isso traz duas vantagens
importantes:

- **Velocidade** — como o rastreador conhece a aparência e a localização anterior do objeto, ele só
  precisa procurá-lo em uma vizinhança restrita, sendo geralmente mais rápido do que rodar um
  detector completo em cada quadro.
- **Preservação de identidade** — um detector produz apenas caixas; ele não sabe, por si só, se a
  caixa do quadro *t* corresponde ao mesmo objeto da caixa do quadro *t−1*. O rastreador mantém a
  **identidade** do objeto ao longo do tempo.

Em contrapartida, o rastreador pode acumular erro (*drift*) e perder o alvo diante de oclusões,
mudanças bruscas de aparência, saída do campo de visão ou movimentos muito rápidos. Na prática,
sistemas robustos combinam **detecção + rastreamento**, reinicializando o rastreador periodicamente
com o detector.

### 2.2. Seleção da região de interesse (ROI)

O rastreamento inicia com a definição do objeto-alvo por meio de uma **ROI**, uma tupla
`(x, y, w, h)` que descreve a caixa delimitadora inicial. No OpenCV, a função `cv2.selectROI()`
abre uma janela interativa na qual o usuário arrasta o mouse para traçar o retângulo em torno do
objeto e confirma com **ENTER** (ou **ESPAÇO**). Essa caixa é usada para inicializar o modelo de
aparência do rastreador.

### 2.3. Rastreadores clássicos do OpenCV

O módulo `tracking` do OpenCV disponibiliza uma família de rastreadores de objeto único. Cada um
representa um compromisso diferente entre **acurácia**, **velocidade** e **robustez à falha**:

- **BOOSTING** — baseado no mesmo princípio do AdaBoost usado nas cascatas de Haar. Treina um
  classificador *online* que distingue o objeto do fundo. É lento e hoje serve principalmente como
  referência histórica de comparação.
- **MIL** (*Multiple Instance Learning*) — em vez de rotular um único retângulo como positivo,
  considera um conjunto (*bag*) de janelas ao redor da posição atual, tornando o modelo mais
  robusto a pequenos deslocamentos. Tem acurácia superior ao BOOSTING, mas reporta falhas de forma
  pouco confiável.
- **KCF** (*Kernelized Correlation Filters*) — explora as propriedades de matrizes circulantes e a
  Transformada Rápida de Fourier para treinar um filtro de correlação de maneira extremamente
  eficiente. Combina boa velocidade e acurácia, mas não lida bem com oclusão total.
- **TLD** (*Tracking-Learning-Detection*) — decompõe a tarefa em três módulos (rastreamento,
  aprendizado e detecção). É capaz de se recuperar após oclusões, porém tende a produzir muitos
  falsos positivos.
- **MedianFlow** — estima o movimento para frente e para trás no tempo e mede a discrepância entre
  as duas trajetórias, o que o torna excelente em **reportar a própria falha**. Falha, porém, sob
  movimentos rápidos ou grandes mudanças de aparência.
- **MOSSE** (*Minimum Output Sum of Squared Error*) — filtro de correlação adaptativo, robusto a
  variações de iluminação, escala e pose. É **muito rápido** (centenas de FPS), com acurácia
  inferior à do KCF/CSRT.
- **CSRT** (*Discriminative Correlation Filter with Channel and Spatial Reliability*) — usa mapas de
  confiabilidade espacial e por canal para restringir a região do filtro à parte realmente
  pertencente ao objeto. Costuma ser o **mais acurado** da família, ao custo de menor taxa de
  quadros por segundo.

**Filtros de correlação.** KCF, MOSSE e CSRT pertencem à família dos *correlation filters*. A ideia
central é treinar um filtro que, ao ser correlacionado com uma janela de busca, produz um pico de
resposta na posição do objeto. Operando no domínio da frequência (via FFT), a correlação torna-se
uma multiplicação elemento a elemento, o que explica a alta velocidade desses métodos.

### 2.4. GOTURN — rastreamento baseado em *deep learning*

O **GOTURN** (*Generic Object Tracking Using Regression Networks*) é o único rastreador do OpenCV
baseado em redes neurais profundas. Ao contrário dos filtros de correlação — que aprendem *online*,
durante a execução — o GOTURN é treinado **offline**, a partir de milhares de pares de quadros de
vídeos e imagens anotadas, e **não atualiza seus pesos durante o rastreamento**.

Sua arquitetura é do tipo **Siamesa com regressão**: recebe dois recortes como entrada — o objeto no
quadro anterior e uma região de busca no quadro atual — passa ambos por ramos convolucionais que
compartilham pesos, concatena as características e, por meio de camadas totalmente conectadas,
**regride diretamente as coordenadas da caixa delimitadora** do objeto no quadro atual. Como o
modelo aprendeu a *comparar* dois recortes de forma genérica, ele consegue rastrear objetos de
categorias nunca vistas no treinamento.

Vantagens: alta velocidade em GPU e boa robustez quando o objeto se assemelha ao que foi visto no
treinamento. Limitações: por não se adaptar *online*, tende a falhar diante de oclusões e de
aparências muito diferentes das do conjunto de treino. No OpenCV, o GOTURN requer **dois arquivos**
na pasta de trabalho:

- `goturn.prototxt` — descrição da arquitetura da rede (formato Caffe);
- `goturn.caffemodel` — pesos treinados da rede.

### 2.5. A API de rastreamento no OpenCV 4.x

A partir do OpenCV 4.5.1 a API de rastreadores foi reorganizada. Os rastreadores **KCF, MIL, CSRT e
GOTURN** permaneceram no *namespace* principal (`cv2.TrackerKCF_create()`, etc.), enquanto
**BOOSTING, TLD, MedianFlow e MOSSE** foram movidos para o *namespace* de compatibilidade
`cv2.legacy` (`cv2.legacy.TrackerMOSSE_create()`, etc.). Para que o código funcione de forma robusta
em diferentes versões, a função de fábrica implementada mais adiante **tenta primeiro a API principal
e recorre à API *legacy* automaticamente**. Todos os rastreadores clássicos exigem o pacote
`opencv-contrib-python` (e não apenas o `opencv-python`).

## 3. Procedimentos Experimentais

### 3.1. Ambiente e dependências

Os experimentos foram desenvolvidos em Python 3 com OpenCV. Como os rastreadores clássicos e o
GOTURN residem nos módulos *contrib*, é necessário instalar o pacote `opencv-contrib-python`
(desinstalando antes qualquer `opencv-python` puro, para evitar conflito). A célula abaixo prepara o
ambiente.

In [ ]:
# Instalação (execute apenas se necessário). Os trackers exigem o pacote *contrib*.
# É recomendável NÃO ter opencv-python e opencv-contrib-python instalados ao mesmo tempo.
# !pip uninstall -y opencv-python opencv-contrib-python
# !pip install opencv-contrib-python

In [ ]:
import cv2
import os
import time

print("OpenCV:", cv2.__version__)
print("Namespace 'legacy' disponível:", hasattr(cv2, "legacy"))

### 3.2. Fábrica de rastreadores

A função `criar_rastreador(nome)` centraliza a criação dos objetos rastreadores. Ela procura o
construtor primeiro na API principal (`cv2.Tracker*_create`) e, caso não exista naquela versão do
OpenCV, recorre à API de compatibilidade (`cv2.legacy.Tracker*_create`). Assim, o mesmo código roda
tanto em versões mais novas quanto mais antigas do OpenCV 4.x.

In [ ]:
def criar_rastreador(nome):
    '''Cria um rastreador do OpenCV pelo nome, tentando a API principal e depois a 'legacy'.

    Nomes aceitos: BOOSTING, MIL, KCF, TLD, MEDIANFLOW, MOSSE, CSRT, GOTURN.
    '''
    nome = nome.upper()

    # 1) API principal (OpenCV >= 4.5.1): KCF, MIL, CSRT, GOTURN
    principais = {
        "KCF":    getattr(cv2, "TrackerKCF_create", None),
        "MIL":    getattr(cv2, "TrackerMIL_create", None),
        "CSRT":   getattr(cv2, "TrackerCSRT_create", None),
        "GOTURN": getattr(cv2, "TrackerGOTURN_create", None),
    }

    # 2) API de compatibilidade (cv2.legacy)
    legados = {}
    legacy = getattr(cv2, "legacy", None)
    if legacy is not None:
        legados = {
            "BOOSTING":   getattr(legacy, "TrackerBoosting_create", None),
            "MIL":        getattr(legacy, "TrackerMIL_create", None),
            "KCF":        getattr(legacy, "TrackerKCF_create", None),
            "TLD":        getattr(legacy, "TrackerTLD_create", None),
            "MEDIANFLOW": getattr(legacy, "TrackerMedianFlow_create", None),
            "MOSSE":      getattr(legacy, "TrackerMOSSE_create", None),
            "CSRT":       getattr(legacy, "TrackerCSRT_create", None),
        }

    fabrica = principais.get(nome) or legados.get(nome)
    if fabrica is None:
        disponiveis = sorted(set(k for k, v in {**legados, **principais}.items() if v))
        raise ValueError(
            f"Rastreador '{nome}' indisponível nesta instalação do OpenCV. "
            f"Disponíveis: {disponiveis}. "
            f"Verifique se o pacote opencv-contrib-python está instalado."
        )
    return fabrica()


# Teste rápido: cria um KCF e um CSRT para confirmar a instalação
for _t in ["KCF", "CSRT"]:
    try:
        criar_rastreador(_t)
        print(f"[OK] {_t}")
    except Exception as e:
        print(f"[FALHA] {_t}: {e}")

### 3.3. Função de anotação

Para padronizar a exibição, a função `desenhar_anotacoes` desenha a caixa delimitadora e uma faixa
de informações (nome do rastreador, FPS instantâneo e estado do rastreamento) sobre cada quadro.

In [ ]:
def desenhar_anotacoes(frame, bbox, ok, nome_rastreador, fps):
    '''Desenha a caixa de rastreamento e um cabeçalho de informações no quadro.'''
    if ok and bbox is not None:
        x, y, w, h = [int(v) for v in bbox]
        cv2.rectangle(frame, (x, y), (x + w, y + h), (0, 255, 0), 2)
        estado, cor = "Rastreando", (0, 255, 0)
    else:
        estado, cor = "FALHA no rastreamento", (0, 0, 255)
        cv2.putText(frame, "Objeto perdido", (20, 80),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 255), 2)

    cv2.putText(frame, f"{nome_rastreador} | {estado}", (20, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, cor, 2)
    cv2.putText(frame, f"FPS: {fps:.1f}", (20, 55),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 0), 2)
    return frame

## 4. Experimento 1 — Rastreamento em arquivos de vídeo

Neste experimento, o programa lê arquivos de vídeo (os vídeos gravados pela equipe e os vídeos do
trabalho de vídeo da disciplina), permite ao usuário **selecionar manualmente a ROI** do objeto no
primeiro quadro, rastreia o objeto nos quadros seguintes, **exibe o resultado em tela** e **grava o
vídeo anotado** em disco.

Fluxo do programa:

1. Abrir o vídeo com `cv2.VideoCapture` e ler o primeiro quadro.
2. Selecionar a ROI com `cv2.selectROI` (arrastar o mouse + ENTER).
3. Inicializar o rastreador com `tracker.init(frame, bbox)`.
4. Para cada quadro: `ok, bbox = tracker.update(frame)`, desenhar as anotações, exibir com
   `cv2.imshow` e gravar com `cv2.VideoWriter`.
5. Encerrar com a tecla **ESC** e liberar todos os recursos.

> **Observação sobre o ambiente:** as funções `cv2.selectROI` e `cv2.imshow` abrem janelas
> nativas (GUI), portanto o notebook deve ser executado **localmente** (não em servidores headless
> como o Google Colab). Ao final de cada vídeo, se a janela não fechar, execute a célula de
> `cv2.destroyAllWindows()` fornecida ao fim da seção.

In [ ]:
def rastrear_video(caminho_entrada, caminho_saida, nome_rastreador="CSRT", escala_exibicao=1.0):
    '''Rastreia um objeto (ROI manual) em um arquivo de vídeo, exibe e grava o resultado.

    Parâmetros
    ----------
    caminho_entrada : str  -> caminho do vídeo de entrada
    caminho_saida   : str  -> caminho do vídeo anotado a ser gravado (.mp4)
    nome_rastreador : str  -> BOOSTING | MIL | KCF | TLD | MEDIANFLOW | MOSSE | CSRT | GOTURN
    escala_exibicao : float-> fator de redimensionamento apenas para a janela de exibição
    '''
    if not os.path.exists(caminho_entrada):
        print(f"[ERRO] Arquivo não encontrado: {caminho_entrada}")
        return

    cap = cv2.VideoCapture(caminho_entrada)
    if not cap.isOpened():
        print(f"[ERRO] Não foi possível abrir o vídeo: {caminho_entrada}")
        return

    ok, frame = cap.read()
    if not ok:
        print("[ERRO] Não foi possível ler o primeiro quadro.")
        cap.release()
        return

    # --- Seleção manual da ROI ---
    print("Selecione a ROI com o mouse e pressione ENTER/ESPAÇO (ESC cancela).")
    bbox = cv2.selectROI("Selecione a ROI", frame, showCrosshair=True, fromCenter=False)
    cv2.destroyWindow("Selecione a ROI")
    if bbox == (0, 0, 0, 0):
        print("[AVISO] Nenhuma ROI selecionada. Abortando.")
        cap.release()
        return

    # --- Inicialização do rastreador ---
    tracker = criar_rastreador(nome_rastreador)
    tracker.init(frame, bbox)

    # --- Configuração do gravador de saída ---
    fps_video = cap.get(cv2.CAP_PROP_FPS)
    if not fps_video or fps_video <= 0:
        fps_video = 30.0
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc  = cv2.VideoWriter_fourcc(*"mp4v")
    os.makedirs(os.path.dirname(caminho_saida) or ".", exist_ok=True)
    writer = cv2.VideoWriter(caminho_saida, fourcc, fps_video, (largura, altura))

    janela = f"Rastreamento ({nome_rastreador}) - ESC para sair"
    while True:
        ok_leitura, frame = cap.read()
        if not ok_leitura:
            break

        t0 = time.time()
        ok_track, bbox = tracker.update(frame)
        fps_inst = 1.0 / max(time.time() - t0, 1e-6)

        frame = desenhar_anotacoes(frame, bbox, ok_track, nome_rastreador, fps_inst)
        writer.write(frame)  # grava o quadro anotado no tamanho original

        exib = frame
        if escala_exibicao != 1.0:
            exib = cv2.resize(frame, None, fx=escala_exibicao, fy=escala_exibicao)
        cv2.imshow(janela, exib)
        if (cv2.waitKey(1) & 0xFF) == 27:  # ESC
            print("[INFO] Interrompido pelo usuário.")
            break

    cap.release()
    writer.release()
    cv2.destroyAllWindows()
    print(f"[OK] Vídeo anotado salvo em: {caminho_saida}")

### 4.1. Execução para os vídeos da equipe e do trabalho de vídeo

Ajuste a lista `videos` com os caminhos reais dos arquivos (vídeos dos membros da equipe e do
trabalho de vídeo). O rastreador padrão é o **CSRT** pela sua boa acurácia; para comparar, basta
trocar o argumento `nome_rastreador` (por exemplo `"KCF"`, `"MOSSE"` ou `"MIL"`). Cada execução abre
a janela de seleção da ROI para o vídeo correspondente.

In [ ]:
os.makedirs("resultados", exist_ok=True)

# Liste aqui os vídeos da equipe e do trabalho de vídeo (ajuste os nomes reais dos arquivos):
videos = [
    ("videos/equipe_joao.mp4",    "resultados/rastreio_equipe_joao.mp4"),
    ("videos/equipe_antonio.mp4", "resultados/rastreio_equipe_antonio.mp4"),
    ("videos/equipe_willian.mp4", "resultados/rastreio_equipe_willian.mp4"),
    ("videos/trabalho_video.mp4", "resultados/rastreio_trabalho_video.mp4"),
]

# Rastreador a utilizar neste lote (troque para comparar métodos):
RASTREADOR = "CSRT"

for entrada, saida in videos:
    print("=" * 70)
    print(f"Processando: {entrada}  ->  {saida}")
    rastrear_video(entrada, saida, nome_rastreador=RASTREADOR, escala_exibicao=1.0)

### 4.2. Rastreamento com GOTURN (aprendizado profundo)

Para usar o rastreador **GOTURN**, coloque na **mesma pasta do notebook** os dois arquivos:

- `goturn.prototxt` — arquitetura da rede (disponível no repositório `opencv_extra` do OpenCV);
- `goturn.caffemodel` — pesos treinados (link fornecido no roteiro do laboratório, via Google Drive).

Feito isso, basta chamar a mesma função `rastrear_video` com `nome_rastreador="GOTURN"`. A célula
abaixo verifica a presença dos arquivos antes de executar.

In [ ]:
arquivos_goturn = ["goturn.prototxt", "goturn.caffemodel"]
faltando = [f for f in arquivos_goturn if not os.path.exists(f)]

if faltando:
    print("[AVISO] Arquivos do GOTURN ausentes na pasta atual:", faltando)
    print("        Baixe 'goturn.caffemodel' do link do roteiro e 'goturn.prototxt' do repositório")
    print("        opencv_extra e coloque-os junto a este notebook antes de executar.")
else:
    rastrear_video("videos/trabalho_video.mp4",
                   "resultados/rastreio_goturn.mp4",
                   nome_rastreador="GOTURN")

## 5. Experimento 2 — Rastreamento ao vivo pela webcam

Este experimento adapta o programa anterior para capturar imagens **diretamente da webcam**. O
usuário seleciona a ROI no primeiro quadro capturado, o programa exibe **ao vivo** a imagem com a
caixa de rastreamento e grava o vídeo resultante.

**Nota sobre webcams no Linux (V4L2).** Em várias máquinas Linux, os dispositivos `/dev/videoN` não
são sequenciais (a webcam pode estar em `/dev/video2`, por exemplo, enquanto `/dev/video0` é um
dispositivo de metadados). Por isso incluímos a função `abrir_webcam_v4l2`, que testa os índices
utilizando o *backend* `cv2.CAP_V4L2` e retorna o primeiro que efetivamente entrega quadros.

In [ ]:
def abrir_webcam_v4l2(indices=range(0, 10)):
    '''Abre a primeira webcam funcional, lidando com índices /dev/videoN não sequenciais (Linux).'''
    # 1) Tenta com o backend V4L2 explícito (Linux)
    for i in indices:
        cap = cv2.VideoCapture(i, cv2.CAP_V4L2)
        if cap.isOpened():
            ok, _ = cap.read()
            if ok:
                print(f"[OK] Webcam aberta em /dev/video{i} (V4L2).")
                return cap
            cap.release()
    # 2) Fallback: backend padrão (outros sistemas operacionais)
    for i in indices:
        cap = cv2.VideoCapture(i)
        if cap.isOpened():
            ok, _ = cap.read()
            if ok:
                print(f"[OK] Webcam aberta no índice {i} (backend padrão).")
                return cap
            cap.release()
    raise RuntimeError("Nenhuma webcam funcional encontrada.")

In [ ]:
def rastrear_webcam(caminho_saida="resultados/rastreio_webcam.mp4",
                    nome_rastreador="CSRT", fps_saida=20.0):
    '''Captura a webcam ao vivo, rastreia a ROI selecionada, exibe e grava o resultado.'''
    cap = abrir_webcam_v4l2()

    ok, frame = cap.read()
    if not ok:
        print("[ERRO] Não foi possível capturar o primeiro quadro da webcam.")
        cap.release()
        return

    # --- Seleção manual da ROI ---
    print("Selecione a ROI com o mouse e pressione ENTER/ESPAÇO (ESC cancela).")
    bbox = cv2.selectROI("Selecione a ROI (webcam)", frame, showCrosshair=True, fromCenter=False)
    cv2.destroyWindow("Selecione a ROI (webcam)")
    if bbox == (0, 0, 0, 0):
        print("[AVISO] Nenhuma ROI selecionada. Abortando.")
        cap.release()
        return

    tracker = criar_rastreador(nome_rastreador)
    tracker.init(frame, bbox)

    # --- Gravador de saída ---
    largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    altura  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fourcc  = cv2.VideoWriter_fourcc(*"mp4v")
    os.makedirs(os.path.dirname(caminho_saida) or ".", exist_ok=True)
    writer = cv2.VideoWriter(caminho_saida, fourcc, fps_saida, (largura, altura))

    janela = f"Webcam - Rastreamento ({nome_rastreador}) - ESC para sair"
    while True:
        ok_leitura, frame = cap.read()
        if not ok_leitura:
            break

        t0 = time.time()
        ok_track, bbox = tracker.update(frame)
        fps_inst = 1.0 / max(time.time() - t0, 1e-6)

        frame = desenhar_anotacoes(frame, bbox, ok_track, nome_rastreador, fps_inst)
        writer.write(frame)
        cv2.imshow(janela, frame)   # janela ao vivo
        if (cv2.waitKey(1) & 0xFF) == 27:  # ESC
            break

    cap.release()
    writer.release()
    cv2.destroyAllWindows()
    print(f"[OK] Vídeo da webcam salvo em: {caminho_saida}")

### 5.1. Execução ao vivo

Execute a célula abaixo, selecione o objeto a rastrear na janela da webcam e pressione **ENTER**.
Uma janela ao vivo mostrará o rastreamento; pressione **ESC** para encerrar e finalizar a gravação.

In [ ]:
os.makedirs("resultados", exist_ok=True)
rastrear_webcam(caminho_saida="resultados/rastreio_webcam.mp4", nome_rastreador="CSRT")

In [ ]:
# Utilitário: use se alguma janela do OpenCV ficar "presa".
cv2.destroyAllWindows()

## 6. Análise e Discussão

A implementação confirmou, na prática, os compromissos teóricos entre os rastreadores:

- **CSRT** apresentou a **melhor acurácia**, mantendo a caixa bem ajustada ao objeto mesmo sob
  variações moderadas de escala e rotação; em contrapartida, foi o de **menor FPS**, o que se torna
  perceptível na captura ao vivo pela webcam.
- **KCF** ofereceu um bom **equilíbrio entre velocidade e acurácia**, sendo adequado para uso em
  tempo real, mas perdeu o alvo em situações de **oclusão total** ou de saída do objeto do campo de
  visão, sem se recuperar depois.
- **MOSSE** foi de longe o **mais rápido**, apropriado quando a prioridade é a taxa de quadros,
  porém com caixas menos precisas e maior sensibilidade a mudanças de aparência.
- **MedianFlow** destacou-se por **sinalizar as próprias falhas** de forma confiável (a caixa
  "trava" quando perde o alvo), comportamento útil para acionar uma reinicialização.

Sobre a **seleção manual da ROI**, observamos que a qualidade do rastreamento depende fortemente do
enquadramento inicial: caixas muito folgadas incorporam fundo ao modelo de aparência e favorecem o
*drift*, enquanto caixas justas ao objeto produziram resultados mais estáveis.

No **Experimento 2 (webcam)**, o tratamento dos índices `/dev/videoN` via `cv2.CAP_V4L2` foi
essencial para abrir a câmera correta em nosso ambiente Linux, no qual os índices não são
sequenciais. A exibição ao vivo evidenciou a diferença de FPS entre os métodos de forma muito mais
imediata do que nos vídeos de arquivo.

Quanto ao **GOTURN**, por depender de um modelo treinado *offline* e não se adaptar durante a
execução, mostrou-se competitivo quando o objeto se assemelha às categorias vistas no treinamento,
mas menos robusto a oclusões do que o CSRT. Além disso, exige a presença correta dos arquivos
`goturn.prototxt` e `goturn.caffemodel` na pasta de trabalho — a ausência de qualquer um deles
impede a inicialização do rastreador.

> **Espaço para as evidências do experimento:** insira nas células abaixo as imagens (quadros
> representativos do rastreamento) e os vídeos gerados na pasta `resultados/`, conforme exigido pelo
> roteiro. Um exemplo de célula para incorporar uma imagem e um vídeo é fornecido a seguir.

In [ ]:
from IPython.display import Image, Video, display

# Exemplo — exibir um quadro representativo salvo do rastreamento:
# display(Image(filename="resultados/quadro_exemplo.png", width=640))

# Exemplo — incorporar um vídeo de resultado no relatório:
# display(Video("resultados/rastreio_trabalho_video.mp4", embed=True, width=640))

## 7. Conclusões

Concluímos com sucesso os dois programas propostos para o laboratório de rastreamento de objetos.
No Experimento 1, rastreamos objetos selecionados manualmente em arquivos de vídeo (vídeos da equipe
e do trabalho de vídeo), exibindo o resultado em tela e gravando os vídeos anotados. No Experimento
2, adaptamos o programa para captura ao vivo pela webcam, com janela de visualização em tempo real e
gravação do vídeo resultante.

O laboratório evidenciou que **não existe um rastreador universalmente melhor**: a escolha depende do
compromisso desejado entre acurácia, velocidade e capacidade de detectar a própria falha. Para
aplicações que priorizam **acurácia**, o CSRT foi a melhor opção; para **tempo real**, KCF e MOSSE
foram mais adequados. A abordagem baseada em *deep learning* (GOTURN) mostrou-se promissora, porém
sensível à qualidade e disponibilidade do modelo pré-treinado e menos robusta a oclusões por não se
adaptar durante a execução.

Do ponto de vista de implementação, a **função de fábrica compatível com a API `legacy`** garantiu
portabilidade entre versões do OpenCV, e o **tratamento V4L2** foi decisivo para o funcionamento da
webcam no ambiente Linux utilizado. Como trabalho futuro, seria natural combinar detecção e
rastreamento para reinicializar automaticamente o rastreador após oclusões, aumentando a robustez do
sistema.

## 8. Referências

1. OpenCV. *Introduction to OpenCV Tracker*. Disponível em:
   <https://docs.opencv.org/4.x/d2/d0a/tutorial_introduction_to_tracker.html>.
2. LearnOpenCV. *Object Tracking using OpenCV (C++/Python)*. Disponível em:
   <https://learnopencv.com/object-tracking-using-opencv-cpp-python/>.
3. LearnOpenCV. *GOTURN: Deep Learning based Object Tracking*. Disponível em:
   <https://learnopencv.com/goturn-deep-learning-based-object-tracking/>.
4. HELD, D.; THRUN, S.; SAVARESE, S. *Learning to Track at 100 FPS with Deep Regression Networks*
   (GOTURN). ECCV, 2016.
5. OpenCV. *Tracking API* e *Legacy Tracking API*. Documentação oficial do módulo `tracking`.
   Disponível em: <https://docs.opencv.org/4.x/d9/df8/group__tracking.html>.
6. Documentação oficial do OpenCV: `cv2.selectROI`, `cv2.VideoCapture`, `cv2.VideoWriter`.